# CUDA-zkML / PCANI - Phase 1-2 validation (roadmap)

Runs `scripts/run_phase12.sh`: clean GPU build, ctest + pytest, host tests of the new R1CS gadgets,
Poseidon, statement v2 and MPC ceremony, independent underconstraint check, Solidity compile, and the
end-to-end GPU experiments E1-E6 (trained exact-ReLU model with context binding, native CONV2D,
batching, private model/input with in-circuit routing, phase-2 ceremony, fully-constrained circuits).

Runtime: **GPU** (any NVIDIA; T4 is fine). Upload the project ZIP when asked. The run is fail-closed;
at the end `Phase12_<gpu>_<date>.zip` is downloaded. Do not edit numbers.

In [ ]:
# 1) Confirm NVIDIA/CUDA runtime
import subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))
print(subprocess.check_output(['nvcc', '--version'], text=True))

In [ ]:
# 2) Upload the project ZIP
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f'Upload exactly one project ZIP; got: {zip_names}')
extract_root = Path('/content/pcani_src')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as zf:
    zf.extractall(extract_root)
repos = [p.parent for p in extract_root.rglob('CMakeLists.txt') if (p.parent / 'scripts' / 'run_gate_f.sh').exists()]
if len(repos) != 1:
    raise RuntimeError(f'Could not identify a unique repository root: {repos}')
REPO = repos[0]
os.chdir(REPO)
print('Repository:', REPO)

In [ ]:
# 3) System packages
!apt-get -qq update
!apt-get -qq install -y cmake build-essential zip > /dev/null
!python -m pip -q install --upgrade pip

In [ ]:
# 4) Run phases 1-2 validation
import subprocess
subprocess.run(['bash', 'scripts/run_phase12.sh'], cwd=REPO, check=True)


In [ ]:
# 5) Summary + download
import json, glob
runs = sorted((REPO / 'results' / 'phase12').iterdir())
summary = json.loads((runs[-1] / 'e2e' / 'phase12_summary.json').read_text())
print(json.dumps(summary['experiments'], indent=2))
print('all passed:', summary['all_passed'])
from google.colab import files
files.download(sorted(glob.glob(str(REPO / 'Phase12_*.zip')))[-1])
